# Inversion Example: FLEXPART Inverse Modeling Training

We will try to run a simple inversion example using FLEXPART and the provided training materials. The goal is to set up the environment, calculate emission sensitivities, and relate emission sources with concentration time series (the inversion process).


We will setup a RELEASE at the Finokalia Atmospheric Observatory (FNAO) in Crete, Greece, and run FLEXPART in backward mode to calculate the emission sensitivities.

You need to prepare your FLEXPART simulation and run it. 

The following steps will guide you through the process.

1. Modify the COMMAND file
2. ~~Modify the RELEASES file~~
3. Modify the OUTGRID file
4. Set the AGECLASS
5. run FLEXPART in backward mode

## Simulation parameters

If you do not remember how to set up the COMMAND, RELEASES, and OUTGRID files, please refer to the FLEXPART documentation: [FLEXPART Documentation about Configuration](https://flexpart.img.univie.ac.at/docs/configuration.html), but most things should be rather intuitive.


COMMAND file tasks:

- We will start on 28.09.2017 with a release of 10.000 airtracer particles for every hour (24 releases). Set the end date at least to 01.10.2017.
- Set the simulation direction to backward.
- Set the output and output average interval to 1 hour.
- Enable convection parameterization.
- Use the option IOUTPUTFOREACHRELEASE=1 and LINIT_COND=1 for backward runs.

OUTGRID tasks:

- Define the lower left corner of the output grid and the upper right corner of the output grid to cover the area of interest (e.g. 10E, 20N, 50E, 50N for a region around Greece).
- Set the output grid to 1 degree horizontal resolution 
- Use 3 vertical levels: 100m, 3000m and 30 km.
- Remember: if you set the output grid too large, you might be outside the domain of the meteorological data and FLEXPART will not be able to run.

Other tasks:

- We will follow every release exactly 1 day (AGECLASS) backward in time. What does that mean? We want particles to terminate after 1 day of backward simulation. This is important for the inversion process, because we want to relate the emission sources with the concentration time series at the receptor site (FNAO).
- build FLEXPART_ETA in [../flexpart](../flexpart/) using `make`
- create the output directory and run FLEXPART in backward mode.


**Before you run FLEXPART, shutdown all other kernels (except this one) or all kernels and run in a terminal or execute the next line** 

In [ ]:
# this should run FLEXPART for you, once you configured all the input files.
!mkdir -p output/ && ../flexpart/FLEXPART_ETA

Do you see: "CONGRATULATIONS: YOU HAVE SUCCESSFULLY COMPLETED A FLEXPART MODEL RUN!" ???

- yes. continue
- no. check the output and try to fix the problem. If you cannot fix it, ask for help.

In [ ]:
# Create a backup of the successful output
!tar cf inversion_output.tar output/

**Download the inversion_output.tar file NOW!** [download here](./inversion_output.tar) as a backup. reupload if your session get's killed! Yes, annoying!

In [ ]:
# Extract the output from a backup
!tar xf inversion_output.tar

# Investigate your results

After the simulation is complete and you read: **"CONGRATULATIONS: YOU HAVE SUCCESSFULLY COMPLETED A FLEXPART MODEL RUN!"** you can start investigating your results.


Tasks:

- locate your grid_time file and provide the path below `grid_time_file = "???"`
- choose different emission source locations and look at the emission sensitivities for these locations.
- check the timeseries to see how the emission sensitivities change over time.

Use emission source values in the range of 0.1 to 10 ng/m²s. 

Questions that you can answer:

- Are certain emission sources responsible for the observed peaks?
- How do the emission sensitivities change over time?

In [ ]:
import functions as fs
############################################## SETTINGS ################################################################
grid_time_file = "output/grid_time_20170929000000.nc" # path to the grid time file
colorbar_limits = [0.1, 1000] # sm³/kg
# height = 100  #[m] height of the level to plot
map_coordinates = [23, 35, 34, 45] # [lon_min, lon_max, lat_min, lat_max]
lat, lon, time, release_times, height, conc, f = fs.read_grid_time_file(grid_time_file)

In [ ]:

########################################## specify emission sources ####################################################
emissions = []
# e.g. use some locations like this:
emissions = fs.add_source(emissions, lat=38.5, lon=29.5, val=1) # Emissions in [ng/m²s]
emissions = fs.add_source(emissions, lat=40.5, lon=27.5, val=5)
emissions = fs.add_source(emissions, lat=39.5, lon=26.5, val=10, edgecolor="red")
emissions = fs.add_source(emissions, lat=43.5, lon=33.5, val=3, edgecolor="yellow")

In [ ]:
#########################################  plot sensitivities   #######################################################
fig=fs.plot_sensitivity_for_all_releases(grid_time_file, map_coordinates, emissions, colorbar_limits)

In [ ]:
#########################################  plot time series  #######################################################
# Compute concentration weighted by emissions in ppt (part per trillion)
timeseries = fs.calculate_timeseries(lat, lon, height, conc, emissions)
ax = fs.plot_timeseries(release_times, timeseries, label="ALL")
ax.legend()
ax.grid()

In [ ]:
#########################################  plot time series  #######################################################
timeseries = fs.calculate_timeseries(lat, lon, height, conc, emissions)
ax = fs.plot_timeseries(release_times, timeseries, label="ALL")
res = []
for emission in range(len(emissions)):
    res.append(fs.calculate_timeseries(lat, lon, height, conc, [emissions[emission]]))
ax.stackplot(release_times,res, labels=[f"S{i}" for i in range(len(emissions))])
ax.legend()
ax.grid()